# Churn Baseline Notebook

**Purpose:** establish and compare a non-ML baseline against a simple ML model on the
provided 12-row sample, per the ML problem-framing memo. This notebook is
illustrative/methodological only — 12 rows is far too small to draw real
performance conclusions. See `ml_problem_framing_memo.md` and
`responsible_data_card.md` for the full framing and caveats before this is
extended to real production data.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import LeaveOneOut
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, accuracy_score

df = pd.read_csv("customer_churn_training.csv")
print(df.shape)
df

(12, 7)


Dataset: 12 customers, 6 features + label. No missing values in this sample.

In [ ]:
print(df['churned'].value_counts())
print()
df.groupby('churned')[['tenure_months','support_tickets','monthly_spend_inr','last_login_days']].mean().round(2)

churned
0    7
1    5
Name: count, dtype: int64


Churned customers in this sample have, on average, much shorter tenure, more
support tickets, lower spend, and far more days since last login — consistent
with intuition, and with why a simple rule performs well below.

In [ ]:
y = df['churned'].values
majority_pred = np.zeros(len(df))

print('Baseline A: majority-class (always predict not-churned)')
print('Accuracy:', accuracy_score(y, majority_pred))
print('Precision:', precision_score(y, majority_pred, zero_division=0))
print('Recall:', recall_score(y, majority_pred, zero_division=0))
print('F1:', f1_score(y, majority_pred, zero_division=0))

Baseline A: majority-class (always predict not-churned)
Accuracy: 0.5833333333333334
Precision: 0.0
Recall: 0.0
F1: 0.0


In [ ]:
# Non-ML business rule: flag risk if inactive > 10 days OR 4+ support tickets
rule_pred = ((df['last_login_days'] > 10) | (df['support_tickets'] >= 4)).astype(int).values

print('Baseline B: simple rule (last_login_days > 10 OR support_tickets >= 4)')
print('Confusion matrix:\n', confusion_matrix(y, rule_pred))
print('Accuracy:', accuracy_score(y, rule_pred))
print('Precision:', precision_score(y, rule_pred, zero_division=0))
print('Recall:', recall_score(y, rule_pred, zero_division=0))
print('F1:', f1_score(y, rule_pred, zero_division=0))

Baseline B: simple rule (last_login_days > 10 OR support_tickets >= 4)
Confusion matrix:
 [[7 0]
 [0 5]]
Accuracy: 1.0
Precision: 1.0
Recall: 1.0
F1: 1.0


The two-line rule already perfectly separates this sample. That is a signal to
be *cautious* about ML, not a green light — see the decision gate in the
framing memo, Section 3.

In [ ]:
features_num = ['tenure_months','support_tickets','monthly_spend_inr','last_login_days']
features_cat = ['plan_type']
X = df[features_num + features_cat]

pre = ColumnTransformer([
    ('num', StandardScaler(), features_num),
    ('cat', OneHotEncoder(drop='first'), features_cat)
])
pipe = Pipeline([('pre', pre), ('clf', LogisticRegression(max_iter=1000))])

loo = LeaveOneOut()
preds = np.zeros(len(df)); probs = np.zeros(len(df))
for train_idx, test_idx in loo.split(X):
    pipe.fit(X.iloc[train_idx], y[train_idx])
    preds[test_idx] = pipe.predict(X.iloc[test_idx])
    probs[test_idx] = pipe.predict_proba(X.iloc[test_idx])[:, 1]

print('ML model: Logistic Regression, Leave-One-Out CV')
print('Confusion matrix:\n', confusion_matrix(y, preds))
print('Accuracy:', accuracy_score(y, preds))
print('Precision:', precision_score(y, preds, zero_division=0))
print('Recall:', recall_score(y, preds, zero_division=0))
print('F1:', f1_score(y, preds, zero_division=0))

ML model: Logistic Regression, Leave-One-Out CV
Confusion matrix:
 [[7 0]
 [0 5]]
Accuracy: 1.0
Precision: 1.0
Recall: 1.0
F1: 1.0


In [ ]:
out = df[['customer_id','churned']].copy()
out['loo_pred_prob'] = probs.round(3)
out['loo_pred_label'] = preds.astype(int)
out

  customer_id  churned  loo_pred_prob  loo_pred_label
0        C001        1          0.939               1
1        C002        0          0.056               0
2        C003        1          0.619               1
3        C004        0          0.006               0
4        C005        0          0.206               0
5        C006        1          0.992               1
6        C007        0          0.051               0
7        C008        1          0.764               1
8        C009        0          0.140               0
9        C010        1          0.742               1
10       C011        0          0.032               0
11       C012        0          0.483               0


Note C012 (0.483) sits closest to the decision boundary — in the framing memo
this is exactly the kind of case the abstention band (0.35–0.55) is designed to
route to a human reviewer rather than auto-score.

In [ ]:
pipe.fit(X, y)
clf = pipe.named_steps['clf']
ohe_names = pipe.named_steps['pre'].named_transformers_['cat'].get_feature_names_out(features_cat)
all_names = features_num + list(ohe_names)
for name, coef in zip(all_names, clf.coef_[0]):
    print(f'{name}: {coef:.3f}')

tenure_months: -0.500
support_tickets: 0.764
monthly_spend_inr: -0.652
last_login_days: 0.748
plan_type_Pro: -0.159
plan_type_Standard: -0.575


## Conclusion

On this sample, the ML model does not beat the simple business rule — both are
perfect, which on 12 rows indicates the sample is trivially separable, not that
either approach generalizes. **Per the decision gate in the framing memo, this
result does not by itself justify moving to an ML system in production.** The
next step is to re-run this exact notebook against a real, larger, time-split
dataset and check whether the ML model's advantage over Baseline B is large
enough, on the business metric defined in the memo, to justify its added
complexity, monitoring, and governance burden (Data Card, Risk Register).